In [ ]:
import os
import random
import time

import numpy as np
import pandas as pd
import pprint

import silence_tensorflow.auto
import tensorflow as tf

import build_model
import experiment_settings
from build_data import build_hurricane_data
from save_model_run import save_model_run
from save_transfer_blueprint import save_transfer_blueprint
from training_instrumentation import TrainingInstrumentation

In [ ]:
__author__ = "Randal J Barnes and Elizabeth A. Barnes"
__version__ = "25 October 2022"

exp_name = "bivariate_normal_101_EPCP24"
DATA_PATH = "../data/"
MODEL_PATH = "saved_models/"
OVERWRITE_MODEL = True

In [ ]:
exp_name = "bivariate_normal_000_EPCP24"

settings = {
    "filename": "nnfit_vlist_02-Jun-2022.dat",
    "uncertainty_type": "bivariate_normal",
    "leadtime": 24,
    "basin": "EP|CP",
    "undersample": False,
    "hiddens": [5, 5],
    "dropout_rate": [0.0, 0.0, 0.0],
    "ridge_param": [0.0, 0.0],
    "learning_rate": 0.001,
    "momentum": 0.9,
    "nesterov": True,
    "batch_size": 64,
    "rng_seed_list": [123],
    "rng_seed": None,
    "act_fun": "relu",
    "n_epochs": 25_000,
    "patience": 50,
    "test_condition": "years",
    "years_test": 2021,
    "val_condition": "random",
    "n_val": 200,
    "n_train": "max",
    "x_names": None,
}

testing_years = 2021
settings["years_test"] = (testing_years,)

rng_seed = 123
settings["rng_seed"] = rng_seed
network_seed = rng_seed

In [ ]:
# --------------------- RUN THE EXPERIMENT ---------------------------
# Build the track data tensors for a bivariate normal model.
(
    data_summary,
    x_train,
    onehot_train,
    x_val,
    onehot_val,
    x_test,
    onehot_test,
    x_valtest,
    onehot_valtest,
    df_train,
    df_val,
    df_test,
    df_valtest,
) = build_hurricane_data(DATA_PATH, settings, verbose=0)

# Define the callbacks
earlystoping_callback = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    mode="min",
    patience=settings["patience"],
    restore_best_weights=True,
    verbose=1,
)

training_callback = TrainingInstrumentation(
    x_train,
    onehot_train,
    interval=50,
)

callbacks = [
    earlystoping_callback,
    # training_callback,
]

# set random seeds
np.random.seed(rng_seed)
random.seed(rng_seed)
tf.random.set_seed(network_seed)

# Create the model name.
model_name = (
    exp_name
    + "_"
    + str(testing_years)
    + "_"
    + settings["uncertainty_type"]
    + "_"
    + f"network_seed_{network_seed}_rng_seed_{settings['rng_seed']}"
)

# Make, compile, and train the model
tf.keras.backend.clear_session()
model = build_model.make_model(
    settings,
    x_train,
    onehot_train,
    model_compile=True,
)
# model.summary()

# check if the model exists
model_savename = MODEL_PATH + model_name + "_weights.h5"

# train the network
pprint.pprint(model_name)
start_time = time.time()
history = model.fit(
    x_train,
    onehot_train,
    validation_data=(x_val, onehot_val),
    batch_size=settings["batch_size"],
    epochs=settings["n_epochs"],
    shuffle=True,
    verbose=0,
    callbacks=callbacks,
)
stop_time = time.time()

# Display the results, and save the model rum.
best_epoch = np.argmin(history.history["val_loss"])
fit_summary = {
    "network_seed": network_seed,
    "elapsed_time": stop_time - start_time,
    "best_epoch": best_epoch,
    "loss_train": history.history["loss"][best_epoch],
    "loss_valid": history.history["val_loss"][best_epoch],
}

In [ ]:
save_model_run(
    data_summary,
    fit_summary,
    model,
    MODEL_PATH,
    model_name,
    settings,
    __version__,
)

save_transfer_blueprint(
    data_summary,
    fit_summary,
    model,
    MODEL_PATH,
    model_name,
    settings,
    __version__,
)

In [ ]:
model.summary()
